In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q segmentation-models-pytorch ultralytics==8.4.149

In [ ]:
from pathlib import Path

import cv2
import json
import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import Dataset, DataLoader
from zipfile import ZipFile

import segmentation_models_pytorch as smp
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
DATASET_URL = (
    "https://github.com/ultralytics/"
    "assets/releases/download/v0.0.0/"
    "crack-seg.zip"
)

DATASETS_ROOT = Path("/content/datasets")
ARCHIVE_PATH = DATASETS_ROOT / "crack-seg.zip"
DATASET_ROOT = DATASETS_ROOT / "crack-seg"

DATASETS_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = ("train", "val", "test")
EXPECTED_COUNTS = {"train": 3717, "val": 200, "test": 112}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

dataset_ready = (DATASET_ROOT / "images" / "train").is_dir()

if not dataset_ready:
    if not ARCHIVE_PATH.is_file():
        torch.hub.download_url_to_file(
            DATASET_URL, str(ARCHIVE_PATH), progress=True
        )

    with ZipFile(ARCHIVE_PATH, "r") as zip_file:
        zip_file.extractall(DATASETS_ROOT)


if not DATASET_ROOT.is_dir():
    candidates = [
        path for path in DATASETS_ROOT.rglob("*")
        if path.is_dir() and (path / "images").is_dir() and "crack" in path.name.lower()
    ]

    if len(candidates) != 1:
        candidates = [
            path for path in DATASETS_ROOT.rglob("images")
            if path.is_dir()
        ]
        if len(candidates) == 1:
            candidates = [candidates[0].parent]


    DATASET_ROOT = candidates[0]

for split in SPLITS:
    required_directories = [
        (DATASET_ROOT / "images" / split),
        (DATASET_ROOT / "labels" / split)
    ]
    
    for directory in (required_directories):
        if not directory.is_dir():
            raise FileExistsError(directory)

print("Dataset extracted:", DATASET_ROOT)

100%|██████████| 91.6M/91.6M [00:00<00:00, 108MB/s] 


Dataset extracted: /content/datasets


In [6]:
DRIVE_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs"
)


DATA_YAML_PATH = (
    DRIVE_OUTPUT_ROOT
    / "block_02" / "crack_seg_local.yaml"
)

TEST_IMAGES_DIR = DATASET_ROOT / "images" / "test"
TEST_LABELS_DIR = DATASET_ROOT / "labels" / "test"

UNET_CHECKPOINT = (
    DRIVE_OUTPUT_ROOT
    / "block_03"
    / "v1_bce_dice"
    / "best.pt"
)

YOLO_CHECKPOINT = (
    DRIVE_OUTPUT_ROOT
    / "block_04"
    / "yolo26n_crack_seg_v1"
    / "weights"
    / "best.pt"
)

OUTPUT_ROOT = DRIVE_OUTPUT_ROOT / "block_06"
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)

**Lock all decisions**

In [7]:
UNET_IMAGE_SIZE = 416
UNET_THRESHOLD = 0.45

YOLO_IMAGE_SIZE = 640
YOLO_AP_CONFIDENCE = 0.001
YOLO_NMS_IOU = 0.70

TEST_SPLIT_USED_FOR_TUNING = False

locked_protocol = {
    "unet_model": "resnet34_unet_bce_dice",
    "unet_image_size": UNET_IMAGE_SIZE,
    "unet_threshold": UNET_THRESHOLD,
    "yolo_checkpoint": str(YOLO_CHECKPOINT),
    "yolo_image_size": YOLO_IMAGE_SIZE,
    "yolo_ap_confidence": YOLO_AP_CONFIDENCE,
    "yolo_nms_iou": YOLO_NMS_IOU,
    "test_split_used_for_tuning": False,
}

protocol_path = OUTPUT_ROOT / "evaluation_protocol.json"

protocol_path.write_text(
    json.dumps(locked_protocol, indent=2), encoding="utf-8"
)

print(json.dumps(locked_protocol, indent=2))

{
  "unet_model": "resnet34_unet_bce_dice",
  "unet_image_size": 416,
  "unet_threshold": 0.45,
  "yolo_checkpoint": "/content/drive/MyDrive/vision_unit_02_outputs/block_04/yolo26n_crack_seg_v1/weights/best.pt",
  "yolo_image_size": 640,
  "yolo_ap_confidence": 0.001,
  "yolo_nms_iou": 0.7,
  "test_split_used_for_tuning": false
}


## U-Net semantic test evaluation

**Polygon → merged semantic mask**


In [8]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp",}


def load_semantic_mask(label_path, image_height, image_width,):
    mask = np.zeros(
        (image_height, image_width,),
        dtype=np.uint8
    )

    if not label_path.is_file():
        return mask

    lines = label_path.read_text(encoding="utf-8",).splitlines()

    for line in lines:
        values = line.strip().split()

        if len(values) < 7:
            continue

        coordinates = np.asarray(
            values[1:], dtype=np.float32
        ).reshape(-1, 2)

        if len(coordinates) < 3:
            continue

        coordinates = np.clip(
            coordinates,
            0.0, 1.0
        )

        coordinates[:, 0] *= image_width
        coordinates[:, 1] *= image_height

        coordinates[:, 0] = np.clip(
            coordinates[:, 0],
            0, image_width - 1
        )

        coordinates[:, 1] = np.clip(
            coordinates[:, 1],
            0, image_height - 1
        )

        polygon = np.rint(coordinates).astype(np.int32)

        cv2.fillPoly(
            mask,
            [polygon],
            color=1,
        )

    return mask

**Test dataset**